In [97]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from ipywidgets import interact, FloatSlider, FloatText, HBox, VBox, Output
import ipywidgets as widgets
from IPython.display import clear_output

stop = False

# Gompertz
def Gompertz(t, k1, k2, k3, k4, k5, k6):
    N_0 = N[0]
    return np.log(N_0) + k1 * np.exp(-np.exp(-k2*(t - k3))) - k4*np.exp(-np.exp(-k5*(t-k6)))
    
#Churchill model
def Churchill(t, k1, k2, k3, k4, k5, k6):
    return ( (1/k1)*np.exp(k3*t) + (1/k2)*np.exp(k4*t) )**(-1)



# Provide an initial guess for the parameters
p0 = [1,1,1,1,1,1]

# Plotting
def plot_fit(k1, k2, k3, k4, k5, k6, func):
    plt.scatter(t, N, label='Data')
    plt.plot(t, np.exp(func(t, k1, k2, k3, k4, k5, k6)), 'r-', label='Fit')
    plt.xlim(0,160)
    plt.ylim(0,8)
    plt.xticks(np.arange(0, 180, step=20))  # Set x-axis ticks
    plt.xlabel('t (days)')
    plt.ylabel('log(CFU/ml)')
    plt.legend()
    #plt.savefig(f'{file_name}_plot.png')  # Save the plot as 'file_name_plot.png'
    plt.show()

#processes file and launches interactables
def process_file(file_name, func):
    clear_output(wait=True)
    global t, N
    data = pd.read_csv(file_name, header=None)
    t = data[0]
    N = data[1]

    # fit
    popt, pcov = curve_fit(func, t, np.log(N),p0=p0, maxfev=9999)

    # Sliders
    k1_slider = FloatSlider(min=0.01, max=100, step=0.001, value=popt[0], format='.4f')
    k2_slider = FloatSlider(min=0.01, max=100, step=0.001, value=popt[1], format='.4f')
    k3_slider = FloatSlider(min=0, max=100, step=0.001, value=popt[2], format='.4f')
    k4_slider = FloatSlider(min=0, max=100, step=0.001, value=popt[3], format='.4f')
    k5_slider = FloatSlider(min=0, max=100, step=0.001, value=popt[4], format='.4f')
    k6_slider = FloatSlider(min=0, max=100, step=0.001, value=popt[5], format='.4f')

    #Typeable part
    k1_text = FloatText(value=popt[0], format='.4f')
    k2_text = FloatText(value=popt[1], format='.4f')
    k3_text = FloatText(value=popt[2], format='.4f')
    k4_text = FloatText(value=popt[3], format='.4f')
    k5_text = FloatText(value=popt[4], format='.4f')
    k6_text = FloatText(value=popt[5], format='.4f')

    # Link the sliders and text boxes
    widgets.jslink((k1_slider, 'value'), (k1_text, 'value'))
    widgets.jslink((k2_slider, 'value'), (k2_text, 'value'))
    widgets.jslink((k3_slider, 'value'), (k3_text, 'value'))
    widgets.jslink((k4_slider, 'value'), (k4_text, 'value'))
    widgets.jslink((k5_slider, 'value'), (k5_text, 'value'))
    widgets.jslink((k6_slider, 'value'), (k6_text, 'value'))

    # update on change
    # Create an output widget
    out = Output()

    # Use the output widget in the interact function
    def update_plot(k1, k2, k3, k4, k5, k6):
        with out:
            clear_output(wait=True)
            plot_fit(k1,k2,k3,k4,k5,k6,func)
    interact(update_plot, k1=k1_slider, k2=k2_slider, k3=k3_slider, k4=k4_slider, k5=k5_slider, k6=k6_slider)

    box = VBox([out], layout=widgets.Layout(height='500px'))

    # Display the output widget
    display(box)
    
#save plot image to .png
def save_file(filename, func):
    k1 = k1_slider.value
    k2 = k2_slider.value
    k3 = k3_slider.value
    k4 = k4_slider.value
    k5 = k5_slider.value
    k6 = k6_slider.value
    plt.scatter(t, N, label='Data')
    plt.plot(t, np.exp(func(t, k1, k2, k3, k4, k5, k6)), 'r-', label='Fit')
    plt.xlabel('t (days)')
    plt.ylabel('log(CFU/ml)')
    plt.legend()
    plt.savefig(filename)  # Save the plot with the given filename

In [98]:
process_file("control.csv", Gompertz)

C:\Users\Kaz\AppData\Local\Temp\ipykernel_35924\3673546254.py:47: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(func, t, np.log(N),p0=p0, maxfev=9999)


interactive(children=(FloatSlider(value=100.0, description='k1', min=0.01, step=0.001), FloatSlider(value=0.01…